## drop long branch samples and duplicates

In [2]:
from Bio import SeqIO
from collections import defaultdict
import csv

In [ ]:
input_fasta = "../data/panand_combined_fast_aligned_trimal_0.5.fasta"
output_fasta = "../data/panand_filtered.fasta"

names_to_remove = [
    'Rhytachne_triaristata_AN21TNTL0354_',
    'Zea_diploperennis_plastid__complete_genome_KR873421.1',
    'Anatherum_bicorne_AN21TSTL0179',
    'Anatherum_ivohibense_AN20T172',
    'Andropogon_distachyos_AN21TSTL0295',
    'Bothriochloa_campii_AN22TNTL169',
    'Eulalia_polyneura_AN20T193',
    'Ischaemum_heterotrichum_AN20T194',
    'Microstegium_nudum_AN20T041',
    'Rottboellia_campestris_AN21TS43',
    'Rottboellia_campestris_MCRTL001',
    'Rottboellia_impressa_AN22TNTL164',
    'Saccharum_aff_villosum_AN21TS64',
    'Themeda_arundinacea_AN21TSTL0296',
    'Tripidium_ravennae_AN21TSTL0292'
]

names_to_remove = set(name.strip() for name in names_to_remove)  # ensure stripped and unique

# Filter sequences
filtered_records = []

for record in SeqIO.parse(input_fasta, "fasta"):
    if record.id.strip() not in names_to_remove:
        filtered_records.append(record)

# Write filtered FASTA
with open(output_fasta, "w") as f:
    SeqIO.write(filtered_records, f, "fasta")

print(f"✅ Filtered FASTA written to {output_fasta}")
print(f"  Removed {len(names_to_remove & set(r.id.strip() for r in SeqIO.parse(input_fasta, 'fasta')))} sequences")

### rename sample names so tip labels on tree will be cleaner; create accession name map csv so original name and final name can be matched

In [ ]:
input_fasta = "../data/panand_filtered.fasta"
output_fasta = "../data/panand_FINAL.fasta"
mapping_file = "../data/panand_accession_name_map.csv"

# Keywords for infraspecific names
INFRASPECIFIC_KEYWORDS = {"var", "subsp", "ssp", "f", "forma", "cv"}

# Track duplicates
name_counts = defaultdict(int)

# Store mapping
mapping = []

# Store new SeqRecords
new_records = []

for record in SeqIO.parse(input_fasta, "fasta"):
    original = record.id.strip()  # Trim spaces

    # Replace dashes except for Coix_lacryma-jobi
    if original.startswith("Coix_lacryma-jobi"):
        cleaned = original
    else:
        cleaned = original.replace("-", "_")

    tokens = cleaned.split("_")

    # Keep genus + species
    new_tokens = tokens[:2]

    # Keep infraspecific info if present
    if len(tokens) > 2 and tokens[2] in INFRASPECIFIC_KEYWORDS and len(tokens) > 3:
        new_tokens.extend(tokens[2:4])

    # Build base name
    base_name = "_".join(new_tokens)

    # Handle duplicates
    name_counts[base_name] += 1
    count = name_counts[base_name]
    final_name = base_name if count == 1 else f"{base_name}-{count}"

    # Save mapping
    mapping.append([original, final_name])

    # Update record
    record.id = final_name
    record.name = final_name
    record.description = ""  # remove description
    new_records.append(record)

# Write renamed FASTA
with open(output_fasta, "w") as f:
    SeqIO.write(new_records, f, "fasta")

# Write mapping CSV
with open(mapping_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["original_name", "final_name"])
    writer.writerows(mapping)

print(f"✅ Renaming complete")
print(f"  Output FASTA: {output_fasta}")
print(f"  Mapping CSV: {mapping_file}")